# Stage 5. Multi-Task Training and Model Comparison

Notebook ini melatih model multi-task dual-path memakai validasi silang lima-fold berbasis pasien, lalu membandingkan lima konfigurasi model untuk membuktikan pilihan desain secara empiris. Perbandingan mencakup jalur fitur (hand-crafted saja, deep embedding saja, atau keduanya) dan backbone deep embedding (ResNet18-CSA lawan MobileNetV3-CSA).

## Environment Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd
import torch

from configs import paths
from src.common import features, manifest as manifest_utils, train
from src.sites.conjunctiva import data

print("device", "cuda" if torch.cuda.is_available() else "cpu")

device cuda


## Load Manifest and Stage 4 Features

Manifest dibangun ulang lalu diberi kolom fold untuk validasi silang lima-fold, stratifikasi per kombinasi dataset dan label anemik. Fitur hand-crafted dan deep embedding ResNet18-CSA dimuat dari hasil Stage 4.

In [2]:
output_dir = paths.outputs_dir("conjunctiva")
artifact_dir = paths.artifacts_dir("conjunctiva")

manifest = manifest_utils.assign_kfold(data.build_manifest(save=False), n_splits=5, seed=42)
handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")
embeddings_resnet18 = np.load(output_dir / "deep_embeddings.npy")
embedding_uids_resnet18 = pd.read_csv(output_dir / "deep_embeddings_uids.csv")["uid"].tolist()

print("manifest", manifest.shape)
print("fold balance:")
print(manifest.groupby(["dataset", "anemic", "fold"]).size().unstack(fill_value=0).to_string())

Eyes-Defy Italy melewati 2 folder tanpa metadata atau file lengkap: [93, 95]
Eyes-Defy India melewati 1 folder tanpa metadata atau file lengkap: [7]
manifest (925, 14)
fold balance:
fold               0   1   2   3   4
dataset   anemic                    
cp_anemic 0       58  57  57  57  57
          1       85  85  85  85  84
eyes_defy 0       25  25  25  25  25
          1       18  18  18  18  18


## Sanity Check: Handcrafted Features with Classical SVM

Sebelum melatih model neural yang lebih kompleks, fitur hand-crafted diuji dengan SVM klasik pada CP-AnemiC mengikuti protokol Paper 1, sebagai bukti bahwa implementasi fitur sudah benar. Akurasi validasi silang seharusnya mendekati 0.849 yang dilaporkan pada literatur.

In [3]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cp_manifest = manifest[manifest["dataset"] == "cp_anemic"].reset_index(drop=True)
cp_handcrafted = handcrafted.set_index("uid").loc[cp_manifest["uid"]].reset_index()
X_sanity = cp_handcrafted[features.HANDCRAFTED_COLUMNS].to_numpy()
y_sanity = cp_manifest["anemic"].to_numpy()

sanity_pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
sanity_grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.01, 0.1]}
sanity_search = GridSearchCV(
    sanity_pipeline, sanity_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="accuracy",
)
sanity_search.fit(X_sanity, y_sanity)
print("best params", sanity_search.best_params_)
print("cross validated accuracy", round(sanity_search.best_score_, 4))
print("target dari Paper 1 (SVM tuned)", 0.849)

best params {'svc__C': 100, 'svc__gamma': 0.1}
cross validated accuracy 0.8225
target dari Paper 1 (SVM tuned) 0.849


## Extract MobileNetV3-CSA Embeddings

Backbone kedua diekstraksi untuk perbandingan langsung dengan ResNet18-CSA. MobileNetV3 lebih ringan dan relevan untuk deployment pada perangkat mobile atau edge.

In [4]:
mobilenet_backbone = features.EmbeddingBackbone(backbone_name="mobilenet_v3_small")
embeddings_mobilenet, embedding_uids_mobilenet = features.extract_deep_embeddings(manifest, model=mobilenet_backbone)
print("mobilenet embeddings shape", embeddings_mobilenet.shape)
print("order matches manifest", list(embedding_uids_mobilenet) == list(manifest["uid"]))

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /home/praktikan/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


  0%|          | 0.00/9.83M [00:00<?, ?B/s]

  8%|▊         | 768k/9.83M [00:00<00:01, 7.57MB/s]

 57%|█████▋    | 5.62M/9.83M [00:00<00:00, 32.7MB/s]

100%|██████████| 9.83M/9.83M [00:00<00:00, 42.3MB/s]

mobilenet embeddings shape (925, 256)
order matches manifest True


## Five Model Configurations

Kelima konfigurasi dilatih dengan protokol validasi silang lima-fold yang identik agar perbandingan adil. Konfigurasi keempat, Full Fusion dengan ResNet18-CSA, adalah kandidat model utama.

In [5]:
configurations = [
    {"name": "Path A only", "use_handcrafted": True, "use_deep": False, "embeddings": None, "embedding_uids": None},
    {"name": "Path B only (ResNet18-CSA)", "use_handcrafted": False, "use_deep": True,
     "embeddings": embeddings_resnet18, "embedding_uids": embedding_uids_resnet18},
    {"name": "Path B only (MobileNetV3-CSA)", "use_handcrafted": False, "use_deep": True,
     "embeddings": embeddings_mobilenet, "embedding_uids": embedding_uids_mobilenet},
    {"name": "Full Fusion (ResNet18-CSA)", "use_handcrafted": True, "use_deep": True,
     "embeddings": embeddings_resnet18, "embedding_uids": embedding_uids_resnet18},
    {"name": "Full Fusion (MobileNetV3-CSA)", "use_handcrafted": True, "use_deep": True,
     "embeddings": embeddings_mobilenet, "embedding_uids": embedding_uids_mobilenet},
]

## Train and Evaluate All Configurations

Bobot loss gabungan memakai nilai tetap, yaitu satu untuk regresi hemoglobin, satu untuk klasifikasi anemia, dan setengah untuk severity ordinal karena severity hanya tersedia pada subset data CP-AnemiC.

In [6]:
results = {}
comparison_rows = []
for configuration in configurations:
    result = train.run_kfold(
        manifest, handcrafted, configuration["embeddings"], configuration["embedding_uids"],
        n_splits=5, epochs=60, batch_size=64, loss_weights=(1.0, 1.0, 0.5),
        use_handcrafted=configuration["use_handcrafted"], use_deep=configuration["use_deep"],
    )
    results[configuration["name"]] = result
    metrics = result["fold_metrics"]
    comparison_rows.append({
        "configuration": configuration["name"],
        "mae_mean": metrics["mae"].mean(),
        "mae_std": metrics["mae"].std(),
        "accuracy_mean": metrics["accuracy"].mean(),
        "accuracy_std": metrics["accuracy"].std(),
        "severity_accuracy_mean": metrics["severity_accuracy"].mean(),
    })
    print("selesai", configuration["name"])

comparison_table = pd.DataFrame(comparison_rows)
print()
print(comparison_table.round(4).to_string(index=False))

selesai Path A only


selesai Path B only (ResNet18-CSA)


selesai Path B only (MobileNetV3-CSA)


selesai Full Fusion (ResNet18-CSA)


selesai Full Fusion (MobileNetV3-CSA)

                configuration  mae_mean  mae_std  accuracy_mean  accuracy_std  severity_accuracy_mean
                  Path A only    1.6323   0.1007         0.6390        0.0731                  0.3042
   Path B only (ResNet18-CSA)    1.6708   0.1110         0.6508        0.0186                  0.3282
Path B only (MobileNetV3-CSA)    1.6944   0.0821         0.6151        0.0169                  0.3197
   Full Fusion (ResNet18-CSA)    1.5739   0.0370         0.6800        0.0303                  0.3085
Full Fusion (MobileNetV3-CSA)    1.6448   0.1134         0.6390        0.0770                  0.2902


## Compare Against Literature Baselines

MAE hemoglobin pada Eyes-Defy pada literatur BPANet sekitar 1.212 g/dL, dan pada CP-AnemiC dengan backbone ViT sekitar 1.50 g/dL. Nilai ini menjadi tolok ukur validitas model yang dilatih di sini, meski dihitung pada seluruh dataset gabungan sehingga tidak sepenuhnya identik protokolnya.

In [7]:
for configuration_name, result in results.items():
    oof = result["oof"].merge(manifest[["uid", "dataset"]], on="uid")
    mae_by_dataset = oof.groupby("dataset").apply(lambda g: np.mean(np.abs(g["hb_pred"] - g["hb_true"])))
    print(configuration_name)
    print(mae_by_dataset.round(4).to_string())
    print()

Path A only
dataset
cp_anemic    1.7579
eyes_defy    1.2185

Path B only (ResNet18-CSA)
dataset
cp_anemic    1.7861
eyes_defy    1.2909

Path B only (MobileNetV3-CSA)
dataset
cp_anemic    1.8152
eyes_defy    1.2962

Full Fusion (ResNet18-CSA)
dataset
cp_anemic    1.6782
eyes_defy    1.2299

Full Fusion (MobileNetV3-CSA)
dataset
cp_anemic    1.7526
eyes_defy    1.2896



## Save Results

Prediksi out-of-fold dan metrik per fold untuk setiap konfigurasi disimpan agar dapat dipakai pada evaluasi mendalam di Stage 6. Checkpoint model per fold untuk konfigurasi Full Fusion ResNet18-CSA, sebagai kandidat utama, disimpan ke folder artifacts.

In [8]:
comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)

for configuration_name, result in results.items():
    slug = configuration_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    result["oof"].to_csv(output_dir / f"multitask_oof_{slug}.csv", index=False)
    result["fold_metrics"].to_csv(output_dir / f"multitask_fold_metrics_{slug}.csv", index=False)

main_candidate = results["Full Fusion (ResNet18-CSA)"]
for fold_index, fold_model in enumerate(main_candidate["models"]):
    checkpoint_path = artifact_dir / f"multitask_full_fusion_resnet18_fold{fold_index}.pt"
    torch.save(fold_model.state_dict(), checkpoint_path)

print("saved comparison table and per-configuration results to", output_dir)
print("saved fold checkpoints for main candidate to", artifact_dir)

saved comparison table and per-configuration results to /home/praktikan/projects/Azril/hemavision/outputs/conjunctiva
saved fold checkpoints for main candidate to /home/praktikan/projects/Azril/hemavision/artifacts/conjunctiva
